# Data preprocessing

Evaluate duplicate records and apply the approved missing-value treatment.
Exploratory findings are documented in `data_exploration.ipynb`.

Run this notebook from the `notebooks/` directory with Python and pandas installed. The input is `../data/raw/student_data.csv`. Each notebook loads the input independently. The previous user environment used Python 3.14.10 and pandas 3.1.1. Execution outputs below record the validation runtime.

### Environment check

The Python interpreter, version, working directory, and pandas version
are displayed to record the execution environment.

In [1]:
import sys
import os
import pandas as pd

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

C:\Users\henri\AppData\Local\Programs\Python\Python314\python.exe
3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]
D:\IMS\Data Mining Project\notebooks
3.1.1


### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

The imported dataset contains 649 rows and 34 columns.
One column is an exported row index, so these are not 34 analytical features.

In [2]:
data = pd.read_csv(
    '../data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)

data.shape

(649, 34)

### Missing values before preprocessing

Missing counts and percentages are recorded before applying the selected treatment.

In [4]:
missing_counts = data.isna().sum()
missing_percentages = (missing_counts / len(data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

,Missing count,Missing (%)
Unnamed: 0,0,0.00
school,1,0.15
sex,10,1.54
age,0,0.00
address,17,2.62
famsize,3,0.46
Pstatus,28,4.31
Medu,27,4.16
Fedu,19,2.93
Mjob,6,0.92


### Treatment of missing absences and failures

A working copy of the dataset is created for preprocessing.
Missing absences and previous failures are replaced with zero,
assuming that blanks represent no absences or failures.
This interpretation remains subject to source confirmation.

In [5]:
student_data_clean = student_data.copy()

student_data_clean['absences'] = student_data_clean['absences'].fillna(0)
student_data_clean['failures'] = student_data_clean['failures'].fillna(0)

### Validation of zero replacement

Before-and-after counts verify the replacement of missing absences and failures with zero.

In [6]:
replacement_check = pd.DataFrame({
    'Missing before': student_data[['absences', 'failures']].isna().sum(),
    'Missing after': student_data_clean[['absences', 'failures']].isna().sum(),
    'Zeros after': student_data_clean[['absences', 'failures']].eq(0).sum()
})

replacement_check

,Missing before,Missing after,Zeros after
absences,260,0,260
failures,551,0,551


### Row preservation and remaining missing values

Row counts and order are checked after filling. Remaining missing values
are displayed for later preprocessing decisions.

In [7]:
assert student_data_clean.shape == student_data.shape
assert student_data_clean.index.equals(student_data.index)
assert student_data_clean[['absences', 'failures']].isna().sum().sum() == 0

student_data_clean.isna().sum()

school         1
sex           10
age            0
address       17
famsize        3
Pstatus       28
Medu          27
Fedu          19
Mjob           6
Fjob          18
reason         7
guardian       8
traveltime    11
studytime     14
failures       0
schoolsup     30
famsup         7
paid          31
activities    31
nursery       20
higher        21
internet       5
romantic       1
famrel        24
freetime      20
goout         31
Dalc          24
Walc          27
health        17
absences       0
G1             0
G2             0
G3             0
dtype: int64

### Working dataset

The prepared table is available as `student_data_clean` in this notebook.
Missing failures and absences are interpreted as zero by user decision.
The source meaning of those blanks remains unconfirmed. Further treatments
require an explicit decision. Duplicate records are evaluated above.